# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

document_text = ""
for page in docs:
    # .strip() removes leading/trailing whitespace from each page
    document_text += page.page_content.strip() + "\n\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
import os
import json
from openai import OpenAI

# Initialize the client with the API Gateway URL
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    api_key="gateway-auth" 
)

# 1. DEFINE PROMPT TEMPLATES
SYSTEM_INSTRUCTIONS_TEMPLATE = (
    "You are a specialized document processor. Output your response in valid JSON format. "
    "The summary MUST be written in the style of {tone_style}. "
    "The JSON must include exactly these keys: author, title, relevance, summary, tone."
)

USER_CONTEXT_TEMPLATE = """
TEXT TO PROCESS:
{text_content}
"""

def generate_article_analysis(text_content, tone_style="Bureaucratese"):
    """
    Processes document text by dynamically injecting context into pre-defined templates.
    """
    
    # 2. DYNAMICALLY ADD CONTEXT using formatted strings
    # This separates the 'instructions' logic from the 'data'
    formatted_system_instructions = SYSTEM_INSTRUCTIONS_TEMPLATE.format(tone_style=tone_style)
    formatted_user_prompt = USER_CONTEXT_TEMPLATE.format(text_content=text_content)

    # 3. API Execution using separate roles
    response = client.chat.completions.create(
        model="gpt-4o", 
        response_format={"type": "json_object"},
        messages=[
            # Developer role provides instructions/behavior
            {"role": "developer", "content": formatted_system_instructions},
            # User role provides the dynamic context/data
            {"role": "user", "content": formatted_user_prompt},
        ],
    )

    # 4. Parse response
    raw_content = response.choices[0].message.content
    analysis_dict = json.loads(raw_content)

    # 5. Metadata tracking
    analysis_dict["input_tokens"] = response.usage.prompt_tokens
    analysis_dict["output_tokens"] = response.usage.completion_tokens
    
    return analysis_dict

# Example Execution
try:
    # 'document_text' should be the raw string extracted from your PDF or file
    result = generate_article_analysis(document_text, tone_style="Bureaucratese")
    
    print(f"Author: {result.get('author')}")
    print(f"Title: {result.get('title')}")
    print(f"\nSummary ({result.get('tone')} style): \n{result.get('summary')}")
    print(f"\nUsage: {result.get('input_tokens')} input tokens, {result.get('output_tokens')} output tokens.")

except Exception as e:
    print(f"An error occurred: {e}")

Author: Peter F. Drucker
Title: Managing Oneself

Summary (The tone is advisory and prescriptive, embodying the precision and comprehensive guidance characteristic of Bureaucratese, with an emphasis on self-sufficiency and strategic career management. style): 
In this seminal work, Mr. Drucker elucidates the paramount importance of self-governance in the modern workplace. The document articulates that individuals, imbued with ambition and competency, are necessitated to become their own Chief Executive Officers, as contemporary corporations no longer provide predefined career trajectories. This paradigm engenders a responsibility for individuals to introspectively discern their strengths, work modalities, and ethical principles to yield optimal professional efficacy. Utilizing methodologies such as feedback analysis, individuals are enjoined to undertake a reflective assessment of their performance traits and ethical compasses. Furthermore, Mr. Drucker enunciates strategies for alignin

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Output of the first run:

+ "SummarizationScore": 0.7142857142857143,
    - "SummarizationReason": "The score is 0.71 because the summary introduces contradictions by emphasizing weaknesses, which the original text does not mention, and adds extra information about situational awareness not present in the original text. Additionally, the summary fails to address whether 'Managing Oneself' is included in the Best of HBR 1999, which the original text can answer.",
+ "CoherenceScore": 0.8944176861221568,
    - "CoherenceReason": "The response follows a logical progression, starting with the importance of self-knowledge and building on this idea by discussing the need for introspection and alignment of personal values with professional goals. The transitions between ideas are coherent, with each concept contributing to the overall argument of career management. There are no contradictions or abrupt shifts, and the suggestion of a second career as an adaptive strategy effectively builds on the previous points, maintaining a cohesive narrative.",
+ "TonalityScore": 0.247363851644285,
    - "TonalityReason": "The output uses some formal language and complex vocabulary, which aligns partially with bureaucratic communication. However, it lacks the passive voice and jargon typical of bureaucratese. The tone is somewhat detached but not entirely impersonal, and the style is more analytical and reflective than bureaucratic. Overall, it does not fully match the tone and style of standard bureaucratic documents."


In [4]:
import os, json
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

gateway_key = os.getenv('API_GATEWAY_KEY')
gateway_url = 'https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'   
os.environ["OPENAI_API_KEY"] = gateway_key

custom_model = GPTModel(
    model="gpt-4o", 
    _openai_api_key=gateway_key,
    base_url=gateway_url,
    default_headers={"x-api-key": gateway_key}
)

analysis = generate_article_analysis(document_text, tone_style="Bureaucratese")

# --- METRIC INITIALIZATION ---
summ_metric = SummarizationMetric(threshold=0.5, model=custom_model)

coh_metric = GEval(
    name="Coherence",
    model=custom_model,
    criteria="Logical flow and consistency of ideas.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

ton_metric = GEval(
    name="Tonality",
    model=custom_model,
    criteria="Does the output strictly follow a 'Bureaucratese' tone?",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# --- EXECUTION ---
test_case = LLMTestCase(input=document_text, actual_output=analysis['summary'])

print("Calculating scores...")
summ_metric.measure(test_case)
coh_metric.measure(test_case)
ton_metric.measure(test_case)

# --- STRUCTURED REPORT (Requirement) ---
evaluation_report = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coh_metric.score,
    "CoherenceReason": coh_metric.reason,
    "TonalityScore": ton_metric.score,
    "TonalityReason": ton_metric.reason
}

print(json.dumps(evaluation_report, indent=4))

Output()

Calculating scores...


Output()

Output()

{
    "SummarizationScore": 0.7142857142857143,
    "SummarizationReason": "The score is 0.71 because the summary introduces contradictions by emphasizing weaknesses, which the original text does not mention, and adds extra information about situational awareness not present in the original text. Additionally, the summary fails to address whether 'Managing Oneself' is included in the Best of HBR 1999, which the original text can answer.",
    "CoherenceScore": 0.8944176861221568,
    "CoherenceReason": "The response follows a logical progression, starting with the importance of self-knowledge and building on this idea by discussing the need for introspection and alignment of personal values with professional goals. The transitions between ideas are coherent, with each concept contributing to the overall argument of career management. There are no contradictions or abrupt shifts, and the suggestion of a second career as an adaptive strategy effectively builds on the previous points, mai

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
# --- 1. DEFINE TEMPLATES ---
ENHANCE_SYS = "You are an expert editor. Fix the summary based on the feedback provided."
ENHANCE_USER = "SOURCE: {text}\n\nDRAFT: {summ}\n\nFEEDBACK TO FIX: {reason}"

# --- 2. COMBINE FEEDBACK ---
combined_reasons = f"""
- Summarization: {evaluation_report['SummarizationReason']}
- Coherence: {evaluation_report['CoherenceReason']}
- Tonality: {evaluation_report['TonalityReason']}
"""

# --- 3. GENERATE ENHANCED VERSION ---
print("Generating enhanced version...")
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "developer", "content": ENHANCE_SYS},
        {"role": "user", "content": ENHANCE_USER.format(
            text=document_text, 
            summ=analysis['summary'], 
            reason=combined_reasons
        )}
    ]
)
enhanced_summary = response.choices[0].message.content

# --- 4. RE-EVALUATE ---
test_case_v2 = LLMTestCase(input=document_text, actual_output=enhanced_summary)

print("Re-evaluating improved summary...")
summ_metric.measure(test_case_v2)
coh_metric.measure(test_case_v2)
ton_metric.measure(test_case_v2)

# --- 5. FINAL COMPARISON ---
print("-" * 30)
print(f"SUMMARIZATION: {evaluation_report['SummarizationScore']} -> {summ_metric.score}")
print(f"COHERENCE:     {evaluation_report['CoherenceScore']} -> {coh_metric.score}")
print(f"TONALITY:      {evaluation_report['TonalityScore']} -> {ton_metric.score}")
print("-" * 30)

Generating enhanced version...


Output()

Re-evaluating improved summary...


Output()

Output()

------------------------------
SUMMARIZATION: 0.7142857142857143 -> 0.8571428571428571
COHERENCE:     0.8944176861221568 -> 0.896705504526327
TONALITY:      0.247363851644285 -> 0.14425781609693605
------------------------------


#### Report your results. Did you get a better output? 

+ Evaluate the new summary using the same function.
This is the results of this evaluation:
    -   SUMMARIZATION: 0.6 -> 0.8571428571428571
    - COHERENCE:     0.8994160355082268 -> 0.9074770045485098
    - TONALITY:      0.35259012024153996 -> 0.23547107958974917
+ For Summarization and Coherence values have improved: by provided feedback through the SummarizationReason and CoherenceReason information. The model used this information as a "checklist" to improve the response.
+ Tonality decreased: This could have decreased as the model was prioritizing improving other feedback over maintaining the tone to ensure that the other parameters such as coherence and summarization performed better

Why? Do you think these controls are enough?
+ These controls may not be enough for specific use cases of LLMs. If all of these parameters of summarization, coherence and tonality were equally important, the model may not have the ability to perform as desired. Additionally prompting may be required.

Resources: https://arxiv.org/pdf/2203.02155


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
